# 13 — Novelty flags, inference & model card

**Research question:** Có thể gắn cờ phạm vi dữ liệu và ràng buộc model/calibrator khi suy luận không?

Baseline: median/IQR-distance trên metadata + geometry, threshold quantile 99% proper-fit, không dùng calibration/test. Đây là heuristic, chưa phải detector OOD có nhãn benchmark.

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Module_01_User_and_Body_Profile.md").exists())
for path in [ROOT, ROOT / "scripts"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import numpy as np
from IPython.display import display, Markdown
report = json.loads((ROOT / "experiments/bodym_phase3/metrics.json").read_text(encoding="utf-8"))


In [2]:
detector = json.loads((ROOT / "models/bodym_phase3/novelty.json").read_text())
print("Novelty fit subjects:", detector["training_subjects"], "features:", len(detector["feature_names"]))
print("Threshold:", detector["threshold"], detector["limitations"])
display(report["ood"])

Novelty fit subjects: 1291 features: 19
Threshold: 2.5193656371895434 Heuristic only: no labeled OOD set, no AUROC or guarantee. Can miss pose, RGB segmentation and domain errors.


{'testA': {'photo_pairs': 1684,
  'flagged_pairs': 22,
  'subjects_with_any_flag': 2,
  'pair_flag_rate': 0.013064133016627079,
  'distance_threshold': 2.5193656371895434,
  'interpretation': 'Novelty heuristic only; no OOD ground-truth or calibrated probability'},
 'testB': {'photo_pairs': 1160,
  'flagged_pairs': 6,
  'subjects_with_any_flag': 2,
  'pair_flag_rate': 0.005172413793103448,
  'distance_threshold': 2.5193656371895434,
  'interpretation': 'Novelty heuristic only; no OOD ground-truth or calibrated probability'}}

In [3]:
from scripts.run_bodym_baseline import read_csv
from src.multimodal.inference import estimate_pair
import torch
torch.set_num_threads(4)
with np.load(ROOT / "data/processed/bodym_phase3/fusion/testB_predictions.npz") as data:
    subject, photo = str(data["subject_ids"][0]), str(data["photo_ids"][0])
    expected = data["prediction"][0].copy()
metadata = {r["subject_id"]: r for r in read_csv(ROOT / "data/raw/bodym/testB/hwg_metadata.csv")}[subject]
quality = json.loads((ROOT / "configs/bodym_phase2.json").read_text())["quality"]
model_dir = ROOT / "models/bodym_phase3"
result = estimate_pair(model_dir / "fusion/model.pt", model_dir / "fusion/calibration.json", model_dir / "novelty.json",
    ROOT / "data/raw/bodym/testB/mask" / (photo + ".png"), ROOT / "data/raw/bodym/testB/mask_left" / (photo + ".png"),
    float(metadata["height_cm"]), float(metadata["weight_kg"]), metadata["gender"], quality)
from src.multimodal.model import load_ablation
from scripts.run_bodym_cnn import tensor_data, predictions
model, checkpoint = load_ablation(model_dir / "fusion/model.pt", "cpu")
with np.load(ROOT / "data/processed/bodym/cnn_128x96_testB.npz") as cached:
    sample = {key: cached[key][:1] for key in ["images", "metadata", "y", "subject_ids", "photo_ids"]}
assert str(sample["photo_ids"][0]) == photo
cpu_cached = predictions(model, tensor_data(sample, checkpoint["normalization"], torch.device("cpu")), checkpoint["normalization"], 1)[0]
actual = np.array([result["estimates"][key]["point_cm"] for key in ["chest", "waist", "hip"]])
np.testing.assert_allclose(actual, cpu_cached, atol=1e-6, rtol=0)
verification = {"sample": "first testB pair", "cpu_raw_cached_max_abs_difference_cm": float(np.abs(actual - cpu_cached).max()),
                "cpu_vs_recorded_batched_backend_max_abs_difference_cm": float(np.abs(actual - expected).max()),
                "note": "Same-backend raw/cache checked at 1e-6 cm. CPU vs recorded GPU/batch arithmetic reported separately; CUDA cuDNN TF32 was enabled in measured GPU run."}
(ROOT / "experiments/bodym_phase3/inference_verification.json").write_text(json.dumps(verification, indent=2) + "\n", encoding="utf-8")
print(verification)
assert result["research_only"] and result["provenance"] == "inferred" and result["requires_review"]
display(result)

{'sample': 'first testB pair', 'cpu_raw_cached_max_abs_difference_cm': 0.0, 'cpu_vs_recorded_batched_backend_max_abs_difference_cm': 0.0024855211611338746, 'note': 'Same-backend raw/cache checked at 1e-6 cm. CPU vs recorded GPU/batch arithmetic reported separately; CUDA cuDNN TF32 was enabled in measured GPU run.'}


{'research_only': True,
 'provenance': 'inferred',
 'verification_status': 'unverified_estimate',
 'recorded_at': '2026-10-21T18:09:23.307857+00:00',
 'variant': 'fusion',
 'checkpoint_sha256': 'b4f22b0152aff8401f19a1a2aa2a583a2a36bb6db2eece3d5c89932cfd62750a',
 'nominal_coverage': 0.9,
 'coverage_scope': 'New subject bundle, all supplied photos. Per-target marginal; joint radius covers all 3 targets together.',
 'uncertainty_limitation': 'Exchangeability required; subject/photo-count/capture domain shift and historical dataset reuse can invalidate guarantees. Not an individual health confidence probability.',
 'quality_flags': {'front': [], 'side': []},
 'ood': {'flagged': False,
  'distance': 0.6587002432205131,
  'threshold': 2.5193656371895434,
  'metadata_range_flag': False,
  'method': 'training-iqr-distance-v1',
  'limitations': 'Heuristic only: no labeled OOD set, no AUROC or guarantee. Can miss pose, RGB segmentation and domain errors.'},
 'requires_review': True,
 'input_qual

In [4]:
display(Markdown((ROOT / "models/bodym_phase3/MODEL_CARD.md").read_text(encoding="utf-8")))

# Model card — BodyM Phase 3 research fusion

## Identity / intended use

Research-only checkpoint `models/bodym_phase3/fusion/model.pt`; trained from scratch on BodyM silhouettes.
Does not replace the selected default `models/bodym_cnn/model.pt`. For reproducible adult body-circumference research, non-commercial use pending license clarification.
Inputs: ordered front/side binary silhouettes, provided height (cm), weight (kg), gender encoded by dataset male/female labels. Outputs: chest/waist/hip (cm).
Does not infer gender, fat percentage, muscle mass, visceral fat or InBody values. Not validated for RGB phone photos, diagnosis, children, pregnancy or clinical planning.

## Dataset / training

Seed 42; proper fit 1291, epoch tuning 404, calibration 323, final refit 1695 subjects.
Calibration subjects excluded from all new fitting, target/metadata normalization, BatchNorm and epoch selection. Split list/hashes/configs saved with experiment.
Labels computed from registered 3D-scan meshes. Multiple photos per person; subject-balanced training/metrics. testA/testB share one subject, reported separately.
Existing architecture/hyperparameters and test sets were explored earlier; the new protocol cannot undo historical adaptive reuse.

## Calibration / uncertainty

Fixed 90% nominal subject-max split conformal. Each calibration subject contributes one maximum residual per target across all their photos; joint score also maximizes across targets.
Theoretical coverage needs exchangeability of subject bundles and an independently fixed predictor. Different photo counts, capture domains and historical selection limit that interpretation here.
Empirical coverage/widths are research metrics, not per-person certainty or clinical tolerance. Calibration is cryptographically bound to its checkpoint; never attach it to the all-train default model.

## Evaluation

Checkpoint SHA256: `b4f22b0152aff8401f19a1a2aa2a583a2a36bb6db2eece3d5c89932cfd62750a`.
Parameters: 141443; selected epoch: 58.

- testA: mean MAE 2.127 cm; joint all-photo subject coverage 88.5%; joint full width 14.864 cm.
- testB: mean MAE 2.636 cm; joint all-photo subject coverage 81.0%; joint full width 14.864 cm.

Full per-target MAE/RMSE/bias, subgroup errors, interval widths and novelty-flag strata: `experiments/bodym_phase3/metrics.json`.
The fusion intervals did not reach the nominal 90% joint coverage on these tests. The uncertainty deployment gate is not met. No radius was retuned on test outcomes.

## Limitations / deployment gate

One seed, no independent RGB/OOD benchmark, no age/ethnicity/device subgroup labels, binary gender labels inherited from source, no validated acceptable-error threshold.
Novelty score is a training-derived heuristic, not a safety classifier. Mask quality flags cannot certify anatomical completeness/pose. Do not suppress flags or treat predictions as observed measurements.
Silhouette/image research needs protected storage and separate processing/research consent before collecting user data. No production authentication, retention or medical safety screening is supplied by this research pipeline.
Requires new consented paired-data evaluation, uncertainty/domain validation and application review flow before user-facing release.

Sources: [BodyM](https://registry.opendata.aws/bodym/), [measurement provenance](https://adversarialbodysim.github.io/), [conformal reference](https://arxiv.org/abs/2107.07511).


## Metrics / error analysis / conclusion

JSON có flagged/unflagged errors/intervals, nhưng không có OOD ground truth nên không báo AUROC hoặc gọi unflagged là an toàn. Quality/novelty flags không được biến mất khi trả prediction; mọi số đo vẫn inferred/unverified. Nhãn gender do người cung cấp, không suy luận từ ảnh. Model card ghi dữ liệu, protocol, intended use và giới hạn.

Còn thiếu: RGB/phone domain validation, nhiều seed, test mới, privacy/consent và safety screening production. Phase 4 tiếp theo là temporal alignment và progress/trend analysis có phân biệt observed/extracted/inferred.